Based on https://github.com/langchain-ai/deep_research_from_scratch/blob/main/notebooks/1_scoping.ipynb

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
model = ChatOllama(
    model="qwen3:32b", 
    temperature=0.0,      # Deterministic for consistent tree search
    top_k=2,             # Limit token sampling diversity
    top_p=0.9,           # Nucleus sampling threshold
    verbose=True,        # Enable detailed LLM logging
    base_url="http://localhost:8080"  # Custom Ollama host
)

In [ ]:
import operator 
from typing_extensions import Optional, Annotated, List, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph import MessagesState
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from prompts import clarify_with_user_instructions, transform_messages_into_research_topic_prompt, lead_researcher_prompt, summarize_webpage_prompt, research_agent_prompt, compress_research_system_prompt, compress_research_human_message

### State defintions

In [ ]:
class AgentInputState(MessagesState):
    """Input state for the full agent - only contains messages from user input."""
    pass

class AgentState(MessagesState):
    """
    Main state for the full multi-agent research system  
    """
    # research brief generated from user conversation history 
    research_brief: Optional[str]
    # messages exchanged with supervisor agent for coordination
    supervisor_messages: Annotated[Sequence[BaseMessage], add_messages]
    # raw unprocessed research notes collected during the research phase 
    raw_notes: Annotated[list[str], operator.add] = []
    # processed and structured notes ready for report generation 
    notes: Annotated[list[str], operator.add] = []
    # final formatted research report 
    final_report: str



### Structured output schemas 

In [ ]:
class ClarifyWithUser(BaseModel):
    """Schema for user clarification decision and questions. """

    need_clarification: bool = Field(
        description="whether the user needs to be asked a clarifying question.",
    ) 
    question: str = Field(
        description="A question to ask the user to clarify the report scope."
    )
    verification: str = Field(
        description="Verify message that we will start research after the user has provided the necessary information."
    )

class ResearchQuestion(BaseModel):
    """Schema for structured research brief generation. """

    research_brief: str = Field(
        description="A research question that will be used to guide the research."
    )

## Functions

### Function get_today_str

In [ ]:
from datetime import datetime

def get_today_str() -> str: 
    """Get current date in a human-readable format."""
    return datetime.now().strftime("%a %b %-d, %Y")

### Define Tavily client

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from tavily import TavilyClient
tavily_client = TavilyClient()

### Function Tavily Search

In [ ]:
summarization_model = ChatOllama(
    model="qwen3:32b", 
    temperature=0.0,      # Deterministic for consistent tree search
    top_k=2,             # Limit token sampling diversity
    top_p=0.9,           # Nucleus sampling threshold
    verbose=True,        # Enable detailed LLM logging
    base_url="http://localhost:8080"  # Custom Ollama host
)

In [ ]:
class Summary(BaseModel):
    """Schema for webpage content summarization"""
    summary: str = Field(description="Concise summary of the webpage content")
    key_excerpts: str = Field(description="Important quotes and excerpts from the content")

In [ ]:
def summarize_webpage_content(webpage_content: str) -> str:
    """Summarize webpage content using the configured summarization model.

    Args:
        webpage_content: Raw webpage content to summarize 
    
    Returns: 
        Formatted summary with key excerpts 
    """
    try: 
        # set up structured output model for summarization 
        structured_model = summarization_model.with_structured_output(Summary)

        summary = structured_model.invoke([
            HumanMessage(content=summarize_webpage_prompt.format(
                webpage_content=webpage_content,
                date=get_today_str()
            ))
        ])

        # Format summary with clear structure
        formatted_summary = (
            f"<summary>\n{summary.summary}\n</summary>\n\n"
            f"<key_excerpts>\n{summary.key_excerpts}\n</key_excerpts>"
        )
        return formatted_summary

    except Exception as e:
        print(f"Failed to summarize webpage: {str(e)}")
        return webpage_content[:1000] + "..." if len(webpage_content) > 1000 else webpage_content

In [ ]:
def deduplicate_search_results(search_results: List[dict]) -> dict:
    """Deduplicate search results by URL to avoid processing duplicate content.

    Args:
        search_results: List of search result dictionaries 

    Returns: 
        Dictionary mapping URLs to unique results 
    """
    unique_results = {}

    for response in search_results: 
        for result in response['results']:
            url = result['url']
            if url not in unique_results:
                unique_results[url] = result
    return unique_results

In [ ]:
def process_search_results(unique_results: dict) -> dict:
    """Process search results by summarizing content where available.

    Args:
        unique_results: Dictionary of unique search results 
    Returns: 
        Dictionary of processed results with summaries
    """
    summarized_results = {}

    for url, result in unique_results.items():
        if not result.get("raw_content"):
            content = result['content']
        else:
            content = summarize_webpage_content(result['raw_content'])
        
        summarized_results[url] = {
            'title': result['title'],
            'content': content
        }

    return summarized_results

In [ ]:
def format_search_output(summarized_results: dict) -> str:
    """Format search results into a well-structured string output
    Args:
        Summarized_results: Dictionary of processed search results 
    Returns:
        Formatted string of search results with clear source separation 
    """
    if not summarized_results: 
        return "No valid search results found. Please try different search queries or use different search API."
    
    formatted_output = "Search results: \n\n"

    for i, (url, result) in enumerate(summarized_results.items(), 1):
        formatted_output += f"\n\n--- SOURCE {i}: {result['fitle']} ---\n"
        formatted_output += f"URL: {url}\n\n"
        formatted_output += f"SUMMARY:\n{result['content']}\n\n"
        formatted_output += "-" * 80 + "\n"

    return formatted_output


In [ ]:

from typing_extensions import Literal
def tavily_search_multiple(
    search_queries: List[str], 
    max_results: int=3, 
    topic: Literal['general', 'news', 'finance'] = 'general',
    include_raw_content: bool = True,
) -> List[dict]:
    """
    Perform search using Tavily API for multiple queries 

    Args: 
        search_queries: List of search queries to execute 
        max_results: Maximum number of results per query 
        topic: Topic filter for search results 
        include_raw_content: Whether to include raw webpage content 

    Returns:
        List of search result dictionaries  
    """

    # Execute searches sequentially 
    search_docs = []
    for query in search_queries: 
        result = tavily_client.search(
            query, 
            max_results=max_results,
            include_raw_content=include_raw_content,
            topic=topic
        )
        search_docs.append(result)
    
    return search_docs

In [ ]:
from langchain_core.tools import InjectedToolArg

#@tool(parse_docstring=True)
@tool
def tavily_search(
    query: str,
    max_results: Annotated[int, InjectedToolArg] = 3,
    topic: Annotated[Literal["general", "news", "finance"], InjectedToolArg] = "general",
) -> str:
    """Fetch results from Tavily search API with content summarization.
    Args:
        query: A single search query to execute
        max_results: Maximum number of results to return 
        topic: Topic to filter results by ('general', 'news', 'finance')

    Returns: 
        Formatted string of search results with summaries
    """
    search_results = tavily_search_multiple(
        [query], 
        max_results=max_results,
        topic=topic,
        include_raw_content=True,
    )

    unique_results = deduplicate_search_results(search_results)
    summarized_results = process_search_results(unique_results)
    return format_search_output(summarized_results)

## Nodes

### Node clarify with user

In [ ]:
from typing_extensions import Literal 
from langgraph.types import Command
from langchain_core.messages import get_buffer_string, HumanMessage, AIMessage
from prompts import clarify_with_user_instructions
from langgraph.graph import START, END, StateGraph

def clarify_with_user(state: AgentState) -> Command[Literal["write_research_brief", "__end__"]]:
    """
    Determine if the user's request contains sufficient information to proceed with research.
    Uses structured output to make deterministic decisions and avoid hallucination.
    Routes to either research brief generation or ends with a clarification question.
    """
    structured_output_model = model.with_structured_output(ClarifyWithUser)

    response = structured_output_model.invoke([
        HumanMessage(content=clarify_with_user_instructions.format(
            messages=get_buffer_string(messages=state["messages"]),
            date=get_today_str()
        ))
    ])

    # Route based on clarification needs
    if response.need_clarification:
        return Command(
            goto=END, 
            update={"messages": [AIMessage(content=response.question)]}
        )
    else:
        return Command(
            goto="write_research_brief", 
            update={"messages": [AIMessage(content=response.verification)]}
        )
    

### Node write_research_brief

In [ ]:
def write_research_brief(state: AgentState):
    """
    Transform the conversation history into a comprehensive research brief.

    Uses structured output to ensure the brief follows the required format
    and contains all the necessary details for effective research. 
    """
    structured_output_model = model.with_structured_output(ResearchQuestion)

    response = structured_output_model.invoke([
        HumanMessage(content=transform_messages_into_research_topic_prompt.format(
            messages=get_buffer_string(state.get("messages", [])),
            date=get_today_str()
        ))
    ])

    return {
        "research_brief": response.research_brief,
        "supervisor_messages": [HumanMessage(content=f"{response.research_brief}")]
    }


In [ ]:
deep_researcher_builder = StateGraph(AgentState, input_schema=AgentInputState)
deep_researcher_builder.add_node("clarify_with_user", clarify_with_user)
deep_researcher_builder.add_node("write_research_brief", write_research_brief)
deep_researcher_builder.add_edge(START, "clarify_with_user")
deep_researcher_builder.add_edge("write_research_brief", END)
scope_research = deep_researcher_builder.compile()

### Node final_report_generation

In [ ]:
from prompts import final_report_generation_prompt

async def final_report_generation(state: AgentState): 
   """
   Final report generation node.
   Synthesizes all research findings into a comprehensive final report.
   """ 
   notes = state.get("notes", [])
   findings = "\n".join(notes)
   final_report_prompt = final_report_generation_prompt.format(
    research_brief=state.get("research_brief", ""), 
    findings=findings,
    date=get_today_str()
   )
   final_report = await model.ainvoke([HumanMessage(content=final_report_prompt)])

   return {
      "final_report": final_report.content,
      "messages": ["Here is the final report: " + final_report.content]
   }


### Node SupervisorState

In [ ]:
from typing_extensions import TypedDict, Annotated

class SupervisorState(TypedDict):
    """
    State for the multi-agent research supervisor.

    Manages coordination between supervisor and research agents, tracking
    research progress and accumulating findings from multiple sub-agents. 
    """
    # Messages exchanged with supervisor for coordinations and decision-making 
    supervisor_messages: Annotated[Sequence[BaseMessage], add_messages]
    # Detailed research brief that guides the overall research direction 
    research_brief: str
    # Processed and structured notes ready for final report generation 
    notes: Annotated[list[str], operator.add] = []
    # Counter tracking the number of research iterations performed 
    research_iterations: int = 0
    # Raw unprocessed research notes collected from sub-agent research 
    raw_notes: Annotated[list[str], operator.add] = []




### Function get_nodes_from_tool_calls

In [ ]:
from langchain_core.messages import BaseMessage, filter_messages

def get_notes_from_tool_calls(messages: List[BaseMessage]) -> list[str]: 
    return [tool_msg.content for tool_msg in filter_messages(messages, include_types="tool")]

try: 
    import nest_asyncio
    try: 
        from IPython import get_ipython
        if get_ipython() is not None: 
            nest_asyncio.apply()
    except ImportError:
        pass
except ImportError:
    pass 


### Tool ConductResearch

In [ ]:

@tool
class ConductResearch(BaseModel):
    """
    Tool for delegating a research task to a specialized sub-agent.  
    """
    research_topic: str = Field(
        description="The topic to research. should be a single topic, and should be described in high details (at least a paragraph)."
    )



### Tool ResearchComplete

In [ ]:
@tool
class ResearchComplete(BaseModel):
    """Tool for indicating that the research process is complete."""
    pass 

### Tool think_tool

In [ ]:
@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """
    Tool for strategic reflection on research progress and decision-making.

    Use this tool after each search to analyze and plan next steps systematically. 
    this creates a deliberate pause in the research workflow for quality decision-making. 

    When to use:
    - After receiving search results: What key information did I find? 
    - Before deciding next steps: Do I have enough to answer comprehensively?
    - When assessing research gaps: What specific information am I still missing? 
    - Before concluding research: Can I provide a complete answer now? 

    Reflection should address: 
    1. Analysis of current findings - What concrete information did I find? 
    2. Gap assessment - What curcial information is still missing? 
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer? 
    4. Strategic decision - Should I continue searching or provide my answer? 

    Args: 
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps 

    Returns: 
        Confirmation that reflection was recorded for decision-making. 
    """
    return f"Reflection recorded: {reflection}"

### Definitions

In [ ]:
max_researcher_iterations = 6
max_concurrent_researchers = 3

### Supervisor Model

In [ ]:
supervisor_tools = [ConductResearch, ResearchComplete, think_tool]
supervisor_model = ChatOllama(
    model="qwen3:32b", 
    temperature=0.0,      # Deterministic for consistent tree search
    top_k=2,             # Limit token sampling diversity
    top_p=0.9,           # Nucleus sampling threshold
    verbose=True,        # Enable detailed LLM logging
    base_url="http://localhost:8080"  # Custom Ollama host
)
supervisor_model_with_tools = supervisor_model.bind_tools(supervisor_tools)

In [ ]:
writer_model = ChatOllama(
    model="qwen3:32b", 
    temperature=0.0,      # Deterministic for consistent tree search
    top_k=2,             # Limit token sampling diversity
    top_p=0.9,           # Nucleus sampling threshold
    verbose=True,        # Enable detailed LLM logging
    base_url="http://localhost:8080"  # Custom Ollama host
)

In [ ]:
tools = [tavily_search, think_tool]
tools_by_name = {tool.name: tool for tool in tools}



In [ ]:
model = ChatOllama(
    model="qwen3:32b", 
    temperature=0.0,      # Deterministic for consistent tree search
    top_k=2,             # Limit token sampling diversity
    top_p=0.9,           # Nucleus sampling threshold
    verbose=True,        # Enable detailed LLM logging
    base_url="http://localhost:8080"  # Custom Ollama host
)

model_with_tools = model.bind_tools(tools)
summarization_model = ChatOllama(
    model="qwen3:32b", 
    temperature=0.0,      # Deterministic for consistent tree search
    top_k=2,             # Limit token sampling diversity
    top_p=0.9,           # Nucleus sampling threshold
    verbose=True,        # Enable detailed LLM logging
    base_url="http://localhost:8080"  # Custom Ollama host
)

compress_model = ChatOllama(
    model="qwen3:32b", 
    temperature=0.0,      # Deterministic for consistent tree search
    top_k=2,             # Limit token sampling diversity
    top_p=0.9,           # Nucleus sampling threshold
    verbose=True,        # Enable detailed LLM logging
    base_url="http://localhost:8080"  # Custom Ollama host
)

In [ ]:
class ResearcherState(TypedDict):
    """
    State for the research agent containing message history and research metadata. 

    this state tracks the researcher's conversation, iteration count for limiting 
    tool calls, the research topic being investigated, compressed findings, 
    and raw research notes for detailed analysis.  
    """
    researcher_messages: Annotated[Sequence[BaseMessage], add_messages]
    tool_call_iterations: int
    research_topic: str
    compressed_research: str
    raw_notes: Annotated[List[str], operator.add]

class ResearcherOutputState(TypedDict):
    """
    Output state for the research agent containing final research results 
    """

In [ ]:
def llm_call(state: ResearcherState):
    """
    Analyze current state and decide on next actions. 
    The model analyzes the current conversation state and decides whther to:
    1. call search tools to gather more information
    2. Provide a final answer based on gathered information 

    Returns updated state with the model's response. 
    """ 
    return {
        "researcher_messages": [
            model_with_tools.invoke(
                [SystemMessage(content=research_agent_prompt)] + state["researcher_messages"]
            )
        ]
    }

In [ ]:
def tool_node(state: ResearcherState):
    """
    Execute all tool calls from the previous LLM response. 
    Returns updated stated with tool execution results.  
    """
    tool_calls = state["researcher_messages"][-1].tool_calls

    # Execute all tool calls 
    observations = []
    for tool_call in tool_calls: 
        tool = tools_by_name[tool_call["name"]]
        observations.append(tool.invoke(tool_call["args"]))

    # Create tool messsage output
    tool_outputs = [
        ToolMessage(
            content=observation,
            name=tool_call["name"],
            tool_call_id=tool_call["id"]
        )
        for observation, tool_call in zip(observations, tool_calls)
    ]
    return {"researcher_messages": tool_outputs}

In [ ]:


def compress_research(state: ResearcherState) -> dict:
    """
    Compress research findings into a concise summary.

    Takes all the research messages and tool outputs and creates 
    a compressed summary suitable for the supervisor's decision-making. 
    """

    system_message = compress_research_system_prompt.format(date=get_today_str())
    messages = [SystemMessage(content=system_message) + state.get("researcher_messages", []) + [HumanMessage(content=compress_research_human_message)]]
    response = compress_model.invoke(messages)

    raw_notes = [
        str(m.content) for m in filter_messages(
            state["researcher_messages"],
            include_types=["tool", "ai"]
        )
    ]

    return {
        "compressed_research": str(response.content),
        "raw_notes": ["\n".join(raw_notes)]
    }

In [ ]:
def should_continue_researcher(state: ResearcherState) -> Literal["tool_node", "compress_research"]:
    """
    Determine whether to continue research of provide final answer. 

    Determine whether the agent should continue hte research loop or provide a final answer
    based on whether the LLM made tool calls. 

    Returns:
        "tool_node": Continue to tool execution 
        "compress_research": Stop and compress research
    """
    messages = state["researcher_messages"]
    last_message = messages[-1]

    if last_message.tool_calls:
        return "tool_node"
    return "compress_research"

In [ ]:
agent_builder = StateGraph(ResearcherState, output_schema=ResearcherOutputState)
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)
agent_builder.add_node("compress_research", compress_research)

agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call", 
    should_continue_researcher,
    {
        "tool_node": "tool_node",
        "compress_research": "compress_research"
    }
)
agent_builder.add_edge("tool_node", "llm_call")
agent_builder.add_edge("compress_research", END)

researcher_agent = agent_builder.compile()

In [ ]:
from langchain_core.messages import SystemMessage, ToolMessage

async def supervisor(state: SupervisorState) -> Command[Literal["supervisor_tools"]]:
    """Coordinate research activities 

    Analyzes the research brief and current progress to decide: 
    - What research topics need investigation 
    - Whether to conduct parallel research 
    - When reserch is complete

    Args: 
        state: Current supervisor state with messages and research progress
    Returns: 
        Command to proceed to supervisor_tools node with updated_state
    """
    supervisor_messages = state.get("supervisor_messages", [])

    system_message = lead_researcher_prompt.format(
        date=get_today_str(), 
        max_concurrent_research_units=max_concurrent_researchers, 
        max_researcher_iterations=max_researcher_iterations
    )
    messages = [SystemMessage(content=system_message)] + supervisor_messages

    # Make decision about next research steps 
    response = await supervisor_model_with_tools.ainvoke(messages)

    return Command(
        goto="supervisor_tools",
        update={
            "supervisor_messages": [response],
            "research_iterations": state.get("research_iterations", 0) + 1
        }
    )

async def supervisor_tools(state: SupervisorState) -> Command[Literal["supervisor", "__end__"]]:
    """Execute supervisor decisions - either conduct research or end the process. 
    
    Handles:
    - Executing_think_tool calls for strategic reflection 
    - Launching parallel research agents for different topics
    - Aggregating research results 
    - determining when research is complete

    Args: 
        state: Current supervisor state with messages and iteration count

    Returns: 
        Command to continue supervision, end process, or handle errors 
    """
    supervisor_messages = state.get("supervisor_messages", [])
    research_iterations = state.get("research_iterations", [])
    most_recent_message = supervisor_messages[-1]

    # Initialize variables for single return pattern 
    tool_messages = []
    all_raw_notes = []
    next_step = "supervisor"
    should_end = False 

    # Check exit criteria first 
    exceeded_iterations = research_iterations >= max_researcher_iterations
    no_tool_calls = not most_recent_message.tool_calls
    research_complete = any(
        tool_call["name"] == "ResearchComplete"
        for tool_call in most_recent_message.tool_calls
    )

    if exceeded_iterations or no_tool_calls or research_complete:
        should_end = True
        next_step = END
    else:
        try:
            # Execute all tool calls from ConductResearch calls 
            think_tool_calls = [
                tool_call for tool_call in most_recent_message.tool_calls
                if tool_call["name"] == "think_tool"
            ]

            conduct_research_calls = [
                tool_call for tool_call in most_recent_message.tool_calls 
                if tool_call["name"] == "ConductResearch"
            ]

            # Handle think_tool calls (synchronous)
            for tool_call in think_tool_calls: 
                observation = think_tool.invoke(tool_call["args"])
                tool_messages.append(
                    ToolMessage(
                        content=observation,
                        name=tool_call["name"],
                        tool_call_id=tool_call["id"]
                    )
                )

            # Handle ConductResearch calls (async)
            if conduct_research_calls:
                # Launch parallel research agents
                coros = [
                    researcher_agent.ainvoke({
                        "researcher_messages": [
                            HumanMessage(content=tool_call["args"]["research_topic"])
                        ],
                        "research_topic": tool_call["args"]["research_topic"]
                    })
                    for tool_call in conduct_research_calls
                ]

                # Wait for all research to complete 
                tool_results = await asyncio.gather(*coros)

                # Format research results as tool messages 
                # Each sub-agent returns compressed reserach findings in result["compressed_research"]
                # We write this compressed reserach as the content of a ToolMessage, which allows
                # the supervisor to later retrieve these findings via get_notes_from_tool_calls()
                research_tool_messages = [
                    ToolMessage(
                        content=result.get("compressed_research", "Error synthesizing research report"),
                        name=tool_call["name"],
                        tool_call_id=tool_call["id"],
                    ) for result, tool_call in zip(tool_results, conduct_research_calls)
                ]

                tool_messages.extend(research_tool_messages)

                # Aggregate raw notes from all research 
                all_raw_notes = [
                    "\n".join(result.get("raw_notes", []))
                    for result in tool_results
                ]

        except Exception as e:
            print(f"Error in supervisor tools: {e}")
            should_end = True
            next_step = END

    if should_end:
        return Command(
            goto=next_step,
            update={
                "notes": get_notes_from_tool_calls(supervisor_messages), 
                "research_brief": state.get("research_brief", "")
            }
        )
    else:
        return Command(
            goto=next_step,
            update={
                "supervisor_messages": tool_messages,
                "raw_notes": all_raw_notes
            }
        )


In [ ]:
# Build supervisor graph 
supervisor_builder = StateGraph(SupervisorState)
supervisor_builder.add_node("supervisor", supervisor)
supervisor_builder.add_node("supervisor_tools", supervisor_tools)
supervisor_builder.add_edge(START, "supervisor")
supervisor_agent = supervisor_builder.compile()

In [ ]:
from langchain_core.messages import HumanMessage 


### Graph Construction 

deep_researcher_builder = StateGraph(AgentState, input_schema=AgentInputState)
deep_researcher_builder.add_node("clarify_with_user", clarify_with_user)
deep_researcher_builder.add_node("write_research_brief", write_research_brief)
deep_researcher_builder.add_node("supervisor_subgraph", supervisor_agent)
deep_researcher_builder.add_node("final_report_generation", final_report_generation)

deep_researcher_builder.add_edge(START, "clarify_with_user")
deep_researcher_builder.add_edge("write_research_brief", "supervisor_subgraph")
deep_researcher_builder.add_edge("supervisor_subgraph", "final_report_generation")
deep_researcher_builder.add_edge("final_report_generation", END)

agent = deep_researcher_builder.compile()

In [ ]:
from rich.console import Console
from rich.panel import Panel 
from rich.text import Text
import json

console = Console()

def format_message_content(message):
    """Convert message content to displayable string"""
    parts = []
    tool_calls_processed = False 

    if isinstance(message.content, str):
        parts.append(message.content)
    elif isinstance(message.content, list):
        for item in message.content:
            if item.get('type') == 'text': 
                parts.append(item['text'])
            elif item.get('type') == 'tool_use': 
                parts.append(f"\n Tool Call: {item['name']}")
                parts.append(f"   Args: {json.dumps(item['input'], indent=2)}")
                parts.append(f"   ID: {item.get['id', 'N/A']}")
                tool_calls_processed = True
    else:
        parts.append(str(message.content))

    if not tool_calls_processed and hasattr(message, 'tool_calls') and message.tool_calls: 
        for tool_call in message.tool_calls: 
            parts.append(f"\n Tool Call: {tool_call['name']}")
            parts.append(f"  Args: {json.dumps(tool_call['args'], indent=2)}")
            parts.append(f"  ID: {tool_call['id']}")

    return "\n".join(parts)

In [ ]:
def format_messages(messages):
    """
    Format and display a list of messages with Rich formatting 
    """
    for m in messages:
        msg_type = m.__class__.__name__.replace("Message", "")
        content = format_message_content(m)

        if msg_type == "Human": 
            console.print(Panel(content, title="Human", border_style="blue"))
        elif msg_type == "Ai": 
            console.print(Panel(content, title="Assistant", border_style="green"))
        elif msg_type == "Tool": 
            console.print(Panel(content, title="Tool Output", border_style="yellow"))
        else: 
            console.print(Panel(content, title=f" {msg_type}", border_style="white"))
        

In [ ]:
from IPython.display import Image, display 
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
full_agent = deep_researcher_builder.compile(checkpointer=checkpointer)
display(Image(full_agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage

thread = {"configurable": {"thread_id": "1", "recursion_limit": 50}}
result = await full_agent.ainvoke(
    {'messages': [HumanMessage(content="Compare Gemini to OpenAI deep research agent in terms of capabilities.")]}, 
    config=thread
    )
format_messages(result['messages'])